# 编译产物 — OM 结构、外置权重、SO in OM、模型缓存

图编译的最终产物是 **OM（Offline Model）离线模型**。本节先剖开 OM 看它内部装了什么，再讲三种面向部署的产物形态配置：**外置权重**（瘦身 OM）、**SO in OM**（算子库随模型打包）、**模型编译缓存**（跳过重复编译）。掌握这些，你才能根据部署约束（OM 体积、环境依赖、冷启动时间）选对产物形态。

本节学习大纲如下：

- OM 文件结构（分区视角）
- OM 的自包含性与使用约束
- 外置权重（External Weight / FileConstant）
- SO in OM（算子 so 随模型打包）
- 模型编译缓存（图编译缓存）
- 三种产物形态对比与组合

> 衔接：3.3 节里我们提到过 `EXTERNAL_WEIGHT`，本节把外置权重展开讲清；模型缓存则与编译耗时直接相关。

## 1. OM 文件结构

OM 不是简单"打包模型"，而是 Graph 经过完整图编译流水线后固化的**自包含二进制**。其文件布局依次为 `ModelFileHeader`、独立的 `ModelPartitionTable` 和各分区数据；分区表记录每个分区的类型、偏移和大小。

### 1.1 分区视角

<p align="center"><img src="./images/om_structure.svg" alt="OM 文件分区结构" width="75%"></p>

| 分区 | 装什么 |
| --- | --- |
| `MODEL_DEF` | 模型定义：图结构、算子属性、各类标记位（如 `so_in_om_flag`） |
| `WEIGHTS_DATA` | 权重数据（外置权重场景下会被搬出，见第 3 节） |
| `TASK_INFO` | 执行任务信息，如 Task 序列和 Stream 编排 |
| `TBE_KERNELS` | 编译后的算子 kernel 二进制 |
| `SO_BINS` | 打包进 OM 的算子 .so（SO in OM 特性，见第 4 节） |
| `TILING_DATA` | 预计算的 tiling 参数 |

> 上表只列出与本节主题直接相关的代表性分区，不是 `ModelPartitionType` 的全量列表。

### 1.2 回顾：OM 是怎么来的

OM 是 Graph 跑完图编译四阶段（图准备 → 图拆分 → 图优化 → 生成执行方案）后的固化结果，把"算子二进制 + 权重 + 执行流（Task/Stream/内存编排）+ 元信息"写进上述分区。

| 组成 | 说明 |
| --- | --- |
| 算子二进制 | 适配昇腾 AI Core 的 kernel |
| 权重数据 | 模型参数 |
| 执行流信息 | Task 序列、Stream 分配、内存编排 |
| 模型元信息 | 输入输出描述、shape、dtype、版本等 |

OM 通过 `aclgrphSaveModel("name", model)` 序列化落盘，文件名自动补 `.om`。

> 说明：若 OM 文件名中带操作系统/架构后缀（如 `xxx_linux_x86_64.om`），则该 OM 只能在对应 OS+架构运行环境使用，需配合 `OPTION_HOST_ENV_OS` / `OPTION_HOST_ENV_CPU` 指定运行环境。

## 2. OM 的自包含性与使用约束

OM 设计为**自包含**：默认情况下，加载一个 OM 就能在昇腾设备上推理，**不再依赖原始 onnx/pb 文件或前端框架运行时**。这正是离线模型能独立分发的根本原因。

| 特性 | 说明 |
| --- | --- |
| 不含原始模型 | OM 里没有 .onnx / .pb，计算已固化成芯片可执行形态 |
| 与芯片绑定 | 为某 `soc_version` 编的 OM 不能直接拿到别的芯片跑，换芯片需重编 |
| 与版本相关 | 跨 CANN 版本的兼容性不保证，升级后建议重新编译 |
| 可独立分发 | 通过 ACL（`aclmdlLoadFromFile` 等）直接加载执行 |

但"自包含"在某些部署场景会带来新问题，于是衍生出三种产物形态配置：

<p align="left"><img src="./images/om_variants.svg" alt="OM 三种产物形态" width="60%"></p>

> 这三者解决的是不同维度的部署痛点（体积 / 依赖 / 时间），可以独立使用，也可以组合使用。下面逐一展开。

## 3. 外置权重（External Weight / FileConstant）

外置权重把模型权重从 OM 或在线编译图中**分离出去、单独存成磁盘文件**。离线保存时可以给 OM "瘦身"；在线场景还可支持同一 Session 内的权重复用。适用场景包括 OM 体积受限、模型需要加密和多模型共享权重。

### 3.1 原理

开启后，编译期会把图中的 `Const`/`Constant` 节点转换为 `FileConstant` 节点：权重数据落盘成独立文件，节点只保存"文件路径 + 偏移 + 大小 + dtype/shape"等元信息。相同权重通过 hash 去重，多个节点/模型可共享同一份权重文件。

离线保存 OM 时，产物形态如下：

```
开启前：OM 文件 = [模型定义 | 权重数据(大) | 算子二进制 | …]
开启后：OM 文件 = [模型定义 | (Const→FileConstant) | 算子二进制 | …] + OM 同级 weight/ 目录
```

在线 GeSession 不一定保存独立 OM 文件，权重目录规则与离线保存不同，见下文。

### 3.2 开启方式

| 方式 | 配置 | 取值 |
| --- | --- | --- |
| 离线（ATC） | `--external_weight` | 0=内嵌（默认），1=外置且不归一（多文件），2=外置且归一（单文件） |
| 离线（Build 接口） | `EXTERNAL_WEIGHT` 编译选项 | 0=内嵌（默认），1=外置且不归一（多文件），2=外置且归一（单文件） |
| 在线（GeSession） | `ge.externalWeight` option | 0=不外置（普通模式默认），1=外置且不归一（多文件），2=外置且归一（单文件） |
| 在线落盘目录 | `ge.externalWeightDir` option | 可选；指定在线外置权重的落盘目录 |

Build 接口示例：

```cpp
{ge::ir_option::EXTERNAL_WEIGHT, "1"}   // 权重外置，按 hash 分文件保存
```

两种外置模式的文件组织方式不同：

- 模式 1：不同权重保存为 `weight_<hash>` 文件，相同权重按 hash 复用。
- 模式 2：所有权重合并为单文件并通过 offset 定位；ATC 命名为 `<--output 文件名>_weight_combined`，Build/GeSession 命名为 `<根图名>_weight_combined`。

落盘目录还取决于编译方式：

- **ATC/Build 保存 OM**：权重文件放在与 OM 同级的 `weight/` 目录。
- **在线 GeSession**：优先使用 `ge.externalWeightDir`；未指定时依次使用 `${ASCEND_WORK_PATH}/tmp_weight_<pid>_<sessionid>` 或当前执行目录下的 `tmp_weight_<pid>_<sessionid>`。

### 3.3 加载约束

外置权重的 OM 在推理加载时要保证能找到权重：

- 用 `aclmdlLoadFromFile` 加载：**权重文件须放在与 OM 同级的 `weight/` 目录**。
- 用 `aclmdlSetConfigOpt` + `aclmdlLoadWithConfig` 加载：对权重目录无强制要求，可在加载时指定。
- `aclmdlSetExternalWeightAddress(handle, weightFileName, devPtr, size)`：把外置权重数据放到用户自管的 Device 内存，按文件名匹配；`size` 需 **32 字节对齐**，且模型执行期间不能释放该内存。

> **默认值说明**：普通在线模式下 `ge.externalWeight` 默认是 `0`；只有在线推理 Hybrid 模式在用户未显式配置时才会内部设为 `1`，以支持同一 Session 内多模型间的权重复用。用户显式配置的 `0`、`1` 或 `2` 不会被该默认行为覆盖。

## 4. SO in OM（算子 so 随模型打包）

传统部署要求目标机安装完整 OPP 算子包（数百个 .so）且版本严格一致。**SO in OM** 把模型实际用到的算子 .so **按需打包进 OM**，使模型文件自带运行时所需算子代码，简化部署、降低版本匹配风险。

### 4.1 解决什么问题

| 传统部署痛点 | SO in OM 的改善 |
| --- | --- |
| 每个节点都装庞大算子包，容器镜像大 | 模型自包含算子 so，无需外部算子包 |
| 编译/运行环境算子版本不一致导致失败 | so 随模型固化，并记录环境信息供加载校验 |
| 动态 shape 运行时编译依赖算子实现 so | tiling/infer shape so 一起打包 |

### 4.2 三类被打包的 SO

GE 用 `MODEL_DEF` 里的 `so_in_om_flag`（uint16，按 bit 标记）记录打包了哪几类 so：

| 类型 | 触发条件（简化） | 用途 |
| --- | --- | --- |
| SpaceRegistry | 模型含动态 shape | RT2 动态 shape 的 infer shape / tiling so |
| OpMasterDevice | TaskDef 含设备端预处理 kernel | 设备端 tiling so |
| Autofuse | 节点含融合算子 bin 路径 | 自动融合算子 so，随模型分发 |

> **版本说明**：上表按本教程的 CANN 9.0.0 基线列出三类 SO。CANN 9.1 的 `SoBinType` 新增了第四类 `kCustomOp`，用于 portable custom op so 的离线保存和加载。

打包采用**按需策略**——只打模型实际用到的算子 so，避免体积浪费。所有 so 序列化写入 OM 的 `SO_BINS` 分区，每个条目带 magic number 校验和类型标记。

### 4.3 加载与执行

模型加载时，`ModelHelper` 解析 `SO_BINS` 分区，按类型分流：SpaceRegistry so 注册到算子实现 registry，OpMasterDevice/Autofuse so 加载到对应缓存；执行时通过 registry 查找并 `dlsym` 调用对应函数。加载阶段还会校验 OM 记录的 OPP 版本 / 编译器版本与当前运行环境是否兼容。

> 用户视角：SO in OM 让"拷一个 OM 文件过去就能跑"成为可能，特别适合**容器化 / 边缘节点**等不便安装完整算子包的部署环境。它由编译流程按依赖自动触发，无需逐算子手工指定。

## 5. 模型编译缓存（图编译缓存）

图编译是计算密集过程（优化 Pass 链、引擎分区、内存规划、任务生成）。模型缓存把编译产物（OM + 变量描述）**持久化到磁盘**，相同图再次编译时直接加载缓存、跳过全流程，显著降低编译耗时——对**服务冷启动、多 Session 共享**尤其有用。

### 5.1 开启方式（两选项联合）

通过两个选项控制，**二者必须同时设置且非空**才生效：

| 选项 | 含义 | 示例 |
| --- | --- | --- |
| `ge.graph_compiler_cache_dir` | 缓存目录（**必须已存在**） | `"./build_cache_dir"` |
| `ge.graph_key` | 图唯一标识（区分不同图的缓存） | `"test_graph_001"` |

GeSession 示例：

```cpp
std::map<ge::AscendString, ge::AscendString> session_options =
    {{"ge.graph_compiler_cache_dir", "./build_cache_dir"}};
auto session = std::make_shared<ge::GeSession>(session_options);

const auto graph = CreateGraph();
std::map<ge::AscendString, ge::AscendString> graph_options =
    {{"ge.graph_key", "test_graph_001"}};
session->AddGraph(0, graph, graph_options);   // 首次编译写缓存，后续命中直接加载
```

> `ge.graph_key` 必须匹配 `^[A-Za-z0-9_\-]{1,128}$`（字母/数字/下划线/连字符，长度 1–128），因为它会作为缓存文件名前缀。

### 5.2 缓存产物

| 文件 | 命名规则 | 作用 |
| --- | --- | --- |
| 索引文件 | `graph_key + ".idx"` | 按 graph_key 快速定位缓存 |
| 模型缓存文件 | `graph_key + 时间戳 + ".om"` | 编译产物本体 |
| 变量格式文件 | `graph_key + 时间戳 + ".rdcpkt"` | 仅图中有变量时生成，用于匹配/失效判断 |
| weight 目录 | 缓存目录下 `weight/` | 开启 `ge.externalWeight=1` 或 `2` 时生成 |

### 5.3 命中条件与失效场景

| 场景 | 行为 |
| --- | --- |
| 缓存不存在 | 正常编译并写入缓存 |
| 缓存存在且匹配 | 直接加载缓存，跳过编译 |
| 图结构变化 | 旧缓存不可用，需手动删缓存文件或换 `graph_key` 重编 |
| 变量格式变化 | 通过 `.rdcpkt` 检测到不匹配，重新触发编译 |
| 跨 CANN 版本 | 兼容性不保证，升级后须清理缓存目录重编 |
| 多进程同 key+dir | 后触发者不实际编译，直接加载先缓存的图（须自行保证 key 唯一性） |

> 调试小技巧：在缓存目录放 `cache.conf` 设 `cache_debug_mode: true`，可"只查找缓存但不加载"（每次仍完整编译），用于验证缓存是否被正确生成。

## 6. 三种产物形态对比与组合

把三种面向部署的产物形态放在一起对比：

| 形态 | 解决什么 | 主要开关 | 收益 | 代价 / 注意 |
| --- | --- | --- | --- | --- |
| 外置权重 | OM 体积大 / 加密 / 多模型共享权重 | `EXTERNAL_WEIGHT` / `ge.externalWeight`（1=多文件，2=单文件） | OM 瘦身、权重可共享/加密 | 加载时要能找到 weight 文件 |
| SO in OM | 目标机无完整算子包 / 版本不一致 | 编译流程按依赖自动触发 | 模型自带算子 so，部署简化 | OM 体积增加；含环境兼容校验 |
| 模型缓存 | 重复编译慢、冷启动慢 | `ge.graph_compiler_cache_dir` + `ge.graph_key` | 跳过重复编译、加速冷启动 | 图/变量/版本变化时缓存失效 |

### 组合使用

这三者可以叠加。例如服务化部署常见组合：**模型缓存 + 外置权重**——

```
GeSession(
   ge.graph_compiler_cache_dir = "/data/cache",
   ge.externalWeight           = "1")
  └ AddGraph(id, graph, { ge.graph_key = "resnet50_v2" })
```

第一次编译：完整编译 → 写缓存 OM + `weight/` 权重目录；服务重启：检测缓存 → 变量描述匹配 → 直接加载，权重从独立文件读取，进一步加速产物序列化与加载。

> 选择心法：**先想清部署约束**——受 OM 体积限制就外置权重；目标环境难装算子包就靠 SO in OM；反复起停、追求冷启动速度就开模型缓存。三者正交，按需取用、可叠加。

## 7. 动手实践：生成真实编译缓存与外置权重

下面构建一张含两个相同 Const 的图，开启 `ge.graph_compiler_cache_dir`、`ge.graph_key` 和 `ge.externalWeight=1`，再创建两个独立 Session 依次在线编译、执行。第一次运行会落盘真实缓存与外置权重，第二次运行可恢复同一份缓存。

单元会为本次执行创建一个无空格的临时缓存目录，并检查实际生成的 `.idx/.om/meta.json/weight_*` 文件。两个 Session 共享本次的新目录，既能验证缓存恢复，也能与其他运行产生的缓存相互隔离。为保证 NumPy 严格对拍不受默认 FP16 降精度影响，示例显式设置 `precision_mode_v2=origin`。运行前需已配置 CANN 环境并可访问 0 号 NPU。

> **耗时提示**：该单元会创建两个独立 Session，分别经历首次编译和缓存恢复，通常需要数十秒或更长；运行时会用一行 `[INFO]` 提示当前正在处理哪个 Session。


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
# === 真机运行：外置权重去重 + 图编译磁盘缓存 + NPU 对拍 ===
import tempfile
import time
from pathlib import Path

import numpy as np
from ge.es.graph_builder import GraphBuilder
from ge.ge_global import GeApi
from ge.graph import Tensor
from ge.graph.types import DataType, Format
from ge.session import Session

DEVICE_ID = 0
GRAPH_ID = 1
GRAPH_KEY = "notebook_external_weight_v2"
SHAPE = [64, 64]

# CANN 编译缓存中包含路径相关信息；为本次实验创建独立的无空格目录，
# 避免不同运行的缓存相互干扰。
cache_dir = Path(tempfile.mkdtemp(prefix="ge_0304_cache_")).resolve()
weight_dir = cache_dir / "weight"
print("本次编译缓存目录：", cache_dir)

weight_array = np.linspace(-1.0, 1.0, num=np.prod(SHAPE), dtype=np.float32).reshape(SHAPE)
input_array = np.ones(SHAPE, dtype=np.float32)
expected = input_array + weight_array + weight_array


def build_weighted_graph():
    builder = GraphBuilder("ExternalWeightGraph")
    x = builder.create_input(
        index=0, name="input_x", data_type=DataType.DT_FLOAT, shape=SHAPE
    )
    # 两个逻辑 Const 内容完全相同，外置时应按内容 hash 复用物理文件。
    weight_values = weight_array.reshape(-1).tolist()
    weight_a = builder.create_const_float(weight_values, shape=SHAPE)
    weight_b = builder.create_const_float(weight_values, shape=SHAPE)
    builder.set_graph_output(x + weight_a + weight_b, 0)
    return builder.build_and_reset()


session_options = {
    "ge.graph_compiler_cache_dir": str(cache_dir),
    "ge.externalWeight": "1",
}
graph_options = {"ge.graph_key": GRAPH_KEY}


def compile_and_run(label, step):
    graph = build_weighted_graph()
    session = None
    operation = "首次编译、写入缓存" if step == 1 else "恢复磁盘缓存"
    print("[进行中 {}/2] {}并在 NPU 上执行...".format(step, operation), flush=True)
    started = time.perf_counter()
    try:
        session = Session(session_options)
        session.add_graph(GRAPH_ID, graph, graph_options)
        input_tensor = Tensor(
            input_array.reshape(-1).tolist(),
            None,
            DataType.DT_FLOAT,
            Format.FORMAT_ND,
            SHAPE,
        )
        outputs = session.run_graph(GRAPH_ID, [input_tensor])
        elapsed_ms = (time.perf_counter() - started) * 1e3

        actual = np.asarray(outputs[0].data, dtype=np.float32)
        np.testing.assert_allclose(actual, expected, rtol=1e-5, atol=1e-5)
        print("{}：在线编译/缓存恢复 + NPU 执行 {:.3f} ms".format(label, elapsed_ms))
        return elapsed_ms
    finally:
        outputs = None
        input_tensor = None
        # 释放 Session 引用，由 Session 析构统一释放图资源。
        session = None


ge_api = GeApi()
ge_api.ge_initialize({
    "ge.exec.deviceId": str(DEVICE_ID),
    "ge.graphRunMode": "0",
    "ge.exec.precision_mode_v2": "origin",
})
try:
    first_ms = compile_and_run("第 1 个 Session", 1)
    second_ms = compile_and_run("第 2 个 Session", 2)
finally:
    ge_api.ge_finalize()

idx_file = cache_dir / (GRAPH_KEY + ".idx")
om_files = sorted(cache_dir.glob("*.om"))
weight_files = sorted(weight_dir.glob("weight_*"))
meta_file = weight_dir / "meta.json"

assert idx_file.is_file(), "未生成模型缓存索引：{}".format(idx_file)
assert om_files, "未生成模型缓存 OM"
assert meta_file.is_file(), "未生成外置权重元数据：{}".format(meta_file)
assert len(weight_files) == 1, "两个相同 Const 应复用一个 weight_* 文件"

print("缓存索引：", idx_file)
print("缓存模型：", [path.name for path in om_files])
print("外置权重：", [path.name for path in weight_files])
print("两次总耗时（仅观察，不假设第二次在所有平台都更快）：{:.3f} / {:.3f} ms".format(
    first_ms, second_ms
))
print("[OK] 真实缓存、外置权重去重与 NPU 数值校验均通过")


## 课后练习

本节讲了 OM 分区结构、自包含性，以及外置权重、SO in OM、模型缓存三种产物形态。请完成以下题目自测。

1. （判断题）默认情况下 OM 是自包含的，加载它就能在昇腾设备上推理，不再依赖原始 onnx/pb 文件或前端框架运行时。

2. （判断题）图编译缓存只需设置 `ge.graph_key` 一个选项即可生效，`ge.graph_compiler_cache_dir` 可以不设。

3. （单选题）开启外置权重后，图中的哪类节点会被转换为 `FileConstant` 类型？
    A. Data 节点
    B. Const / Constant 节点
    C. Add 等计算节点
    D. 图的输出节点

4. （单选题）以下关于 SO in OM 特性的描述，哪个是正确的？
    A. 它把模型用到的算子 .so 按需打包进 OM 的 SO_BINS 分区
    B. 它会把所有 OPP 算子包里的 so 全部打进 OM
    C. 它的作用是把权重从 OM 中分离出去
    D. 它只在静态 shape 模型中触发，动态 shape 不打包

5. （多选题）以下关于图编译缓存的描述，哪些是正确的？
    A. `ge.graph_compiler_cache_dir` 和 `ge.graph_key` 必须同时设置且非空才生效
    B. 缓存目录必须已存在，否则会编译失败
    C. 图结构变化后旧缓存不可用，需删缓存或换 graph_key 重编
    D. 跨 CANN 版本的缓存一定可以直接复用

6. （多选题）以下关于 OM 文件的描述，哪些是正确的？
    A. OM 依次包含文件头、独立分区表和分区数据；权重分区是 WEIGHTS_DATA，算子二进制分区是 TBE_KERNELS
    B. OM 与目标芯片版本绑定，换芯片需重新编译
    C. OM 中包含原始的 .onnx / .pb 文件以便回溯
    D. 通过 ATC/Build 保存外置权重 OM 时，权重文件位于 OM 同级的 weight/ 目录

7. （单选题）服务化部署中，为加速冷启动并给 OM 瘦身，推荐的组合是？
    A. 关闭所有融合 + 关闭内存复用
    B. 模型缓存 + 外置权重
    C. 仅 SO in OM
    D. `PRECISION_MODE_V2=origin` + 关闭 buffer 优化

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.04_answer.txt